# we read all the 213 featrues from table and turn it into clinic_feature.pt

In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

df = pd.read_excel("clinic_feat_all.xlsx")

df["性别_bin"] = df["性别"].map({"男": 1, "女": 0}).astype("float32")

age_scaler = StandardScaler()
df["年龄_norm"] = age_scaler.fit_transform(df[["年龄"]])

clinic_cols = (
    ["性别_bin", "年龄_norm"] +
    [f"feat_{i}" for i in range(211)]
)

missing = [c for c in clinic_cols if c not in df.columns]
if len(missing) > 0:
    raise ValueError(f"Missing columns: {missing}")

clinic_dict = {
    str(row["id"]): row[clinic_cols].values.astype("float32")
    for _, row in df.iterrows()
}

torch.save(clinic_dict, "clinic_features.pt")

example_pid = str(df.iloc[0]["id"])
print("clinic_features.pt saved")
print(clinic_dict[example_pid].shape)

# two stage training 
## Run Name Consistency Across Stages

The `run_name` must be **identical for Stage 1 and Stage 2** training.

- **Stage 1 (`train_stage1.py`)**  
  All cross-validated Attention MIL training artifacts are saved under  
  `outputs/stage1_mil/{run_name}/`, including:
  - trained Attention MIL heads
  - corresponding train / validation patient IDs for each fold
  - train log

- **Stage 2 (`train_stage2.py`)**  
  Uses the same `run_name` to:
  - load the **pretrained Attention MIL head** from Stage 1
  - read the **exact same train / validation patient splits**
  - keep the Attention MIL head **frozen**
  - train only the **linear FiLM fusion layer** and **classifier head**
    
  All Stage-2 outputs are saved under  
  `outputs/stage2_film/{run_name}/`.

This ensures **strict fold consistency** and **leakage-free multimodal training** across stages.


In [ ]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 50 \
  --batch_size 4 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0.5 \
  --run_name run3

In [ ]:
%run train_stage2.py \
  --feat_dir ./encoder_features \
  --labels_csv ./labels.xlsx \
  --clinic_pt ./clinic_features.pt \
  --label_cols 内分泌代谢疾病 \
  --max_feats 16 \
  --epochs 70 \
  --batch_size 4 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --folds 5 \
  --instance_strategy random \
  --architecture attention \
  --use_combined_loss \
  --auc_weight 0 \
  --run_name run3